# Patrol Staff Deployment Optimization POC

This notebook builds a **district × watch patrol staffing optimization model** from the attached `Patrol_POC.xlsx` file.

It covers:

- data loading and feature engineering
- the mixed-integer optimization model
- baseline vs optimized staffing results
- charts and maps for operational interpretation


In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import plotly.express as px

ROOT = Path.cwd().resolve().parents[0]
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from patrol_staffing.data_prep import load_calls, build_features, build_map_data
from patrol_staffing.optimization import optimize_staffing

DATA_PATH = ROOT / "data" / "raw" / "Patrol_POC.xlsx"
calls = load_calls(DATA_PATH)
calls.shape

(1084, 152)

## Optimization model

For each district-watch cell \(i\):

- \(x_i\): integer number of officers assigned
- \(s_i\): shortage slack if staffing is below the required level
- \(d_i^+, d_i^-\): deviation from the baseline staffing proxy

### Objective

\[
\min \sum_i \left(0.5x_i + w_is_i + 0.15d_i^+ + 0.15d_i^- \right)
\]

This objective balances:
- shortage reduction, especially in high-priority cells
- staff efficiency
- realism, by avoiding unnecessary movement away from current staffing

### Constraints

1. **Coverage requirement**
\[
x_i + s_i \ge r_i
\]

2. **Movement accounting**
\[
x_i - d_i^+ + d_i^- = b_i
\]

3. **District-watch movement limits**
\[
b_i - 1 \le x_i \le b_i + 1
\]

4. **Watch-level balance**
Each watch total can move by at most ±2 officers.

5. **System budget**
Total staffing can increase by at most 6 flex officers above baseline.


In [2]:
feat = build_features(calls)
result, summary = optimize_staffing(feat)
map_df = build_map_data(calls)

summary

{'records_modeled': 697, 'district_watch_cells': 12, 'baseline_staff_total': 54, 'optimized_staff_total': 60, 'baseline_shortage_total': 69.36, 'optimized_shortage_total': 63.36, 'baseline_weighted_shortage_total': 297.27, 'optimized_weighted_shortage_total': 270.37, 'shortage_reduction_pct': 8.7, 'weighted_shortage_reduction_pct': 9.0}

In [3]:
result[[
    "district","watch","patrol_calls","p1","p2","p3",
    "baseline_staff","optimized_staff","staff_change",
    "required_units","baseline_shortage","optimized_shortage"
]].sort_values(["watch","district"])

 district     watch  patrol_calls  p1  p2  p3  baseline_staff  optimized_staff  staff_change  required_units  baseline_shortage  optimized_shortage
        1 1st Watch            74  26  32   7               4                5             1       15.918155          11.918155           10.918155
        2 1st Watch            65  24  30   2               5                6             1       15.462996          10.462996            9.462996
        3 1st Watch            57  22  20   3               3                3             0       10.456746           7.456746            7.456746
        4 1st Watch            45  22  20   2               4                4             0        8.125000           4.125000            4.125000
        1 2nd Watch            61  17  28   8               5                6             1        9.069841           4.069841            3.069841
        2 2nd Watch            67  15  31  15               5                6             1       11.015972    

In [4]:
heat = result.pivot(index="district", columns="watch", values="staff_change")[["1st Watch","2nd Watch","3rd Watch"]]
px.imshow(heat, text_auto=True, aspect="auto",
          title="Recommended staff changes by district and watch",
          labels=dict(x="Watch", y="District", color="Staff change"))

In [5]:
comparison = pd.DataFrame({
    "Scenario": ["Baseline", "Optimized"],
    "Weighted shortage": [summary["baseline_weighted_shortage_total"], summary["optimized_weighted_shortage_total"]],
    "Shortage": [summary["baseline_shortage_total"], summary["optimized_shortage_total"]],
})
px.bar(comparison.melt(id_vars="Scenario", var_name="Metric", value_name="Value"),
       x="Scenario", y="Value", color="Metric", barmode="group",
       title="Baseline vs optimized shortages")

In [6]:
px.scatter_map(
    map_df,
    lat="latitude",
    lon="longitude",
    size="patrol_calls",
    color="p1_calls",
    hover_name="district",
    hover_data={"avg_service_mins":":.1f", "latitude":False, "longitude":False},
    zoom=10,
    map_style="carto-positron",
    title="District call intensity map"
)

## Results snapshot

Using the uploaded file, the model found:

- **697** patrol-dispatched calls modeled
- **54** baseline staff slots
- **60** optimized staff slots
- **8.7%** reduction in total shortage
- **9.0%** reduction in weighted shortage

### Main operational recommendation
Increase capacity mostly in **Districts 1 and 2 across all watches**, add one officer in **District 3 / 3rd Watch**, and trim **District 4 / 3rd Watch** by one slot.


## Next improvements

1. Replace the baseline staffing proxy with the actual roster or scheduled headcount.
2. Move from district-watch to district-hour optimization.
3. Add adjacency and travel-time constraints.
4. Add service-level constraints for priority 1 calls.
5. Link staffing changes to actual response-time outcomes.
6. Extend to scenario-based or robust optimization using forecast demand.
